<center><img src="images/logo.png" alt="drawing" width="400" style="background-color:white; padding:1em;" /></center> <br/>

# Application of Deep Learning to Text and Image Data
## Module 2, Lab 5: Fine-Tuning BERT


BERT stands for Bidirectional Encoder Representations from Transformers. To learn how BERT works, you will fine-tune the BERT model to classify product reviews. You will use a library called *transformers* to download a pretrained BERT model.

You will learn how to do the following:

- Load and format a dataset.
- Load a pretrained BERT model.
- Train and test a BERT model.

**Important:** BERT and its variants use more resources than the models you have used so far. This might cause your instance to run out of memory. If that happens, do the following:
1. Restart the kernel. (Choose **Kernel** > **Restart Kernel**.)
2. Reduce the batch size.
3. Rerun the code.

__Note:__ You will use a light version of the original BERT implementation called DistilBERT. For more information, see the [DistilBERT, a Distilled Version of BERT: Smaller, Faster, Cheaper, and Lighter](https://doi.org/10.48550/arXiv.1910.01108) paper from 2020 by Victor Sanh, Lysandre Debut, Julien Chaumond, and Thomas Wolf.

---

This lab uses a dataset from a small sample of Amazon product reviews.

__Dataset schema:__
* __reviewText:__ Text of the review
* __summary:__ Summary of the review
* __verified:__ Whether the purchase was verified (true or false)
* __time:__ Unix timestamp for the review
* __log\_votes:__ Logarithm-adjusted votes, log(1+votes)
* __isPositive:__ Whether the review is positive or negative (1 or 0)


---

You will be presented with two kinds of exercises throughout the notebook: activities and challenges. <br/>

| <img style="float: center;" src="images/activity.png" alt="Activity" width="125"/>| <img style="float: center;" src="images/challenge.png" alt="Challenge" width="125"/>|
| --- | --- |
|<p style="text-align:center;">No coding is needed for an activity. You try to understand a concept, <br/>answer questions, or run a code cell.</p> |<p style="text-align:center;">Challenges are where you can practice your coding skills.</p> |


---
## Index

* [Reading and formatting the dataset](#Reading-and-formatting-the-dataset)
* [Loading the pretrained model](#Loading-the-pretrained-model)
* [Training and testing the model](#Training-and-testing-the-model)

---
## Initial Setup

In [1]:
%%capture
!pip install  scikit-learn>=1.1.3 transformers>=4.5.1 datasets>=2.1.0 tokenizers>=0.11.1 evaluate>=0.3.0 huggingface_hub>=0.13.0 xxhash>=3.2.0 d2l>=0.17.0 torch>=1.12.0 torchtext>=0.12.0 matplotlib>=3.3.4 seaborn>=0.11.2 numpy>=1.22.4 pandas>=1.5.2 nltk>=3.8.1

To restart the kernel after installing the libraries, run the following cell.

In [2]:
from IPython.display import HTML, display
def restart_kernel_and_run_all_cells():
    display(HTML(
        '''
            <script>
                code_show = false;
                var cell_idx = Jupyter.notebook.get_cell_elements().index(cell_element);
                cell_idx++;
                function restart(){
                    IPython.notebook.kernel.restart();
                    IPython.notebook.execute_cells([cell_idx])
                }
                restart()
            </script>
        '''
    ))
#print("Before you continue, wait until the kernel is ready again.")
restart_kernel_and_run_all_cells()
print("You are ready to go to the next cell!")

You are ready to go to the next cell!


---
## Reading and formatting the dataset

First, you need to read in the product review dataset and prepare it for the BERT model. To keep the training time low, you will use only the first 2,000 data points from the dataset.

After you understand how to train the model, if you want to improve it, you can use more data to train a new model.

In [3]:
import os
import sys
import time
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
from torch.utils.data import DataLoader

# Set tokenizer parallelism to false
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [8]:
from google.colab import drive
drive.mount('/content/drive')

import sys
import os

# List files in the current directory
files = !ls

# If the file is not in the current directory, you need to find its path
if 'MLUDTI_EN_M2_Lab5_quiz_questions.py' not in files:
    module_path = os.path.abspath(os.path.join('/content/drive/MyDrive/Colab Notebooks/CCC/Deep Learning'))
    if module_path not in sys.path:
        sys.path.append(module_path)


# Import utility functions that provide answers to challenges
from MLUDTI_EN_M2_Lab5_quiz_questions import *

Mounted at /content/drive


Read the dataset.

In [4]:
df = pd.read_csv("/content/NLP-REVIEW-DATA-CLASSIFICATION-TRAINING.csv")

Print the dataset information to see the field types.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56000 entries, 0 to 55999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ID          56000 non-null  int64  
 1   reviewText  55989 non-null  object 
 2   summary     55987 non-null  object 
 3   verified    56000 non-null  bool   
 4   time        56000 non-null  int64  
 5   log_votes   56000 non-null  float64
 6   isPositive  56000 non-null  int64  
dtypes: bool(1), float64(1), int64(3), object(2)
memory usage: 2.6+ MB


You only want data points that contain __reviewText__, so drop the rows that don't have this information.

In [6]:
df.dropna(subset=["reviewText"], inplace=True)

<div style="border: 4px solid coral; text-align: center; margin: auto;">
    <h3><i>Try it yourself!</i></h3>
    <br>
    <p style="text-align:center;margin:auto;"><img src="images/activity.png" alt="Activity" width="100" /> </p>
    <p style=" text-align: center; margin: auto;">Let's remember the business problem: Sentiment analisys over customer reviews.</p>
    <p style=" text-align: center; margin: auto;">To test your understanding of the dataset used for that, run the following cell.</p>
    <br>
</div>

In [9]:
question_1

BERT requires a lot of compute power for large datasets. To reduce the amount of time that it takes to train the model, you will use only the first 2,000 data points for this lab.

In [10]:
df = df.head(2000)

Now, split the dataset into training and validation datasets, keeping 10 percent for validation.

In [11]:
# Separates 10 percent of the entire dataset into the validation dataset
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["reviewText"].tolist(),
    df["isPositive"].tolist(),
    test_size=0.10,
    shuffle=True,
    random_state=324,
    stratify = df["isPositive"].tolist(),
)

You need to tokenize the data. To tokenize the training and validation texts, use a special tokenizer that was built for the DistilBERT model.

In [12]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(train_texts,
                            truncation=True,
                            padding=True)
val_encodings = tokenizer(val_texts,
                          truncation=True,
                          padding=True)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Create a new `ReviewDataset` class to use with the BERT model. Later, you will use the training and validation encoding-label pairs with this new class.

In [13]:
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]).to(device) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx]).to(device)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ReviewDataset(train_encodings, train_labels)
val_dataset = ReviewDataset(val_encodings, val_labels)

---
## Loading the pretrained model

Now, you need to load the model. When you do this, several warnings will print about the last classification layer of BERT, where you are using a randomly initialized layer. You can ignore the warnings because they aren't relevant to the type of training that you are doing.

In [14]:
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased",
                                                            num_labels=2)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['pre_classifier.weight', 'classifier.weight', 'pre_classifier.bias', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


The last step is to freeze all weights until the last classification layer in the BERT model. This helps accelerate the training process. Training the weights of the whole network (66 million weights) would take a long time. Additionally, 2,000 data points would not be enough for that task.

Instead, the following code freezes all the weights until the last classification layer. This means that only a small portion of the weights gets updated (the remaining stay the same). This is a common practice with large language models.

In [15]:
# Freeze the encoder weights until the classifier
for name, param in model.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False

---
## Training and testing the model

Now that the data is ready and you have configured the model, it's time to start the fine-tuning process. This code would take a *long time* (30 minutes or more) to complete with a large dataset, which is why you are running it on a subset of the full review dataset.

First, define the accuracy function.

In [16]:
def calculate_accuracy(output, label):
    """Calculate the accuracy of the trained network.
    output: (batch_size, num_output) float32 tensor
    label: (batch_size, ) int32 tensor """

    return (output.argmax(axis=1) == label.float()).float().mean()

Now you need to create the training and validation loop. This loop will be similar to the previous train/validation loops; however, a few extra parameters are needed because of the transformer architecture.

You need to use the `attention_mask` argument and get the loss from the output of the model with `loss = output[0]`.

In [17]:
# Hyperparameters
num_epochs = 10
learning_rate = 0.005

# Get the compute device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create data loaders
train_loader = DataLoader(train_dataset, shuffle=True,
                          batch_size=16, drop_last=True)
validation_loader = DataLoader(val_dataset, batch_size=8,
                               drop_last=True)

# Set up the optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

model = model.to(device)

for epoch in range(num_epochs):

    train_loss, val_loss, train_acc, valid_acc = 0., 0., 0., 0.

    start = time.time()
    # Training loop starts
    model.train() # put the model in training mode
    for batch in train_loader:
        # Zero the parameter gradients
        optimizer.zero_grad()
        # Put data, label, and attention mask to the correct device
        data = batch["input_ids"].to(device)
        attention_mask = batch['attention_mask'].to(device)
        label = batch["labels"].to(device)

        # Make forward pass
        output = model(data, attention_mask=attention_mask, labels=label)

        # Calculate the loss (this comes from the output)
        loss = output[0]
        # Make backward pass (calculate gradients)
        loss.backward()
        # Accumulate training accuracy and loss
        train_acc += calculate_accuracy(output.logits, label).item()
        train_loss += loss.item()
        # Update weights
        optimizer.step()

    # Validation loop
    # This loop tests the trained network on the validation dataset
    # No weight updates here
    # torch.no_grad() reduces memory usage when not training the network
    model.eval() # Activate evaluation mode
    with torch.no_grad():
        for batch in validation_loader:
            data = batch["input_ids"].to(device)
            attention_mask = batch['attention_mask'].to(device)
            label = batch["labels"].to(device)
            # Make forward pass with the trained model so far
            output = model(data, attention_mask=attention_mask, labels=label)
            # Accumulate validation accuracy and loss
            valid_acc += calculate_accuracy(output.logits, label).item()
            val_loss += output[0].item()

    # Take averages
    train_loss /= len(train_loader)
    train_acc /= len(train_loader)
    val_loss /= len(validation_loader)
    valid_acc /= len(validation_loader)

    end = time.time()

    print("Epoch %d: train loss %.3f, train acc %.3f, val loss %.3f, val acc %.3f, seconds % .3f " % (
        epoch+1, train_loss, train_acc, val_loss, valid_acc, end-start))

Epoch 1: train loss 0.658, train acc 0.616, val loss 0.634, val acc 0.620, seconds  31.503 
Epoch 2: train loss 0.630, train acc 0.631, val loss 0.608, val acc 0.665, seconds  31.739 
Epoch 3: train loss 0.609, train acc 0.655, val loss 0.583, val acc 0.690, seconds  32.298 
Epoch 4: train loss 0.580, train acc 0.693, val loss 0.555, val acc 0.675, seconds  32.845 
Epoch 5: train loss 0.562, train acc 0.724, val loss 0.532, val acc 0.675, seconds  33.978 
Epoch 6: train loss 0.536, train acc 0.749, val loss 0.505, val acc 0.780, seconds  34.818 
Epoch 7: train loss 0.512, train acc 0.777, val loss 0.482, val acc 0.795, seconds  35.156 
Epoch 8: train loss 0.493, train acc 0.781, val loss 0.466, val acc 0.835, seconds  35.841 
Epoch 9: train loss 0.472, train acc 0.800, val loss 0.444, val acc 0.805, seconds  35.473 
Epoch 10: train loss 0.461, train acc 0.799, val loss 0.424, val acc 0.840, seconds  35.978 


### Observe what is happening

The fine-tuned BERT model is able to correctly classify the sentiment of most of the records in the validation set.

Now you will observe in more detail how the sentences are tokenized and encoded by choosing one sentence as an example.

In [18]:
st = val_texts[19]
print(f"Sentence: {st}")
tok = tokenizer(st, truncation=True, padding=True)
print(f"Encoded Sentence: {tok['input_ids']}")

Sentence: An excellent resource for all scanner owners.  Seems to be up to date and all inclusive.  I highly recommend this product!
Encoded Sentence: [101, 2019, 6581, 7692, 2005, 2035, 26221, 5608, 1012, 3849, 2000, 2022, 2039, 2000, 3058, 1998, 2035, 18678, 1012, 1045, 3811, 16755, 2023, 4031, 999, 102]


Print the vocabulary size.

In [19]:
# The mapped vocabulary is stored in tokenizer.vocab
tokenizer.vocab_size

30522

Use the encoded sentence with the tokenizer to recover the original sentence.

In [20]:
# Methods convert_ids_to_tokens and convert_tokens_to_ids allow you to see how sentences are tokenized
print(tokenizer.convert_ids_to_tokens(tok["input_ids"]))

['[CLS]', 'an', 'excellent', 'resource', 'for', 'all', 'scanner', 'owners', '.', 'seems', 'to', 'be', 'up', 'to', 'date', 'and', 'all', 'inclusive', '.', 'i', 'highly', 'recommend', 'this', 'product', '!', '[SEP]']


### Get predictions on the test data

After the model is trained, you can focus on getting test data to make predictions with. First, you will read and format the test dataset. Then, you will pass the test data to the trained model and make predictions.

In [21]:
# Read the test data (It doesn't have the isPositive label.)
df_test = pd.read_csv("/content/NLP-REVIEW-DATA-CLASSIFICATION-TRAINING.csv")
df_test.head()

,ID,reviewText,summary,verified,time,log_votes,isPositive
0,65886,Purchased as a quick fix for a needed Server 2...,"Easy install, seamless migration",True,1458864000,0.000000,1
1,19822,So far so good. Installation was simple. And r...,Five Stars,True,1417478400,0.000000,1
2,14558,Microsoft keeps making Visual Studio better. I...,This is the best development tool I've ever used.,False,1252886400,0.000000,1
3,39708,Very good product.,Very good product.,True,1458604800,0.000000,1
4,8015,So very different from my last version and I a...,... from my last version and I am having a gre...,True,1454716800,2.197225,0


As you did previously, drop the rows that don't have __reviewText__.

In [22]:
df_test.dropna(subset=["reviewText"], inplace=True)

Making predictions will take a long time with this model. To get results quickly, start by only making predictions with 15 data points from the test set.

In [23]:
test_texts = df_test["reviewText"].tolist()[0:15]

In [24]:
test_encodings = tokenizer(test_texts,
                           truncation=True,
                           padding=True)

Create labels for the test dataset to pass zeros by using `[0]*len(test_texts)`.

In [25]:
test_dataset = ReviewDataset(test_encodings, [0]*len(test_texts))

Then, create a data loader for the test set and record the corresponding predictions.

In [26]:
test_loader = DataLoader(test_dataset, batch_size=4)
test_predictions = []
model.eval()

with torch.no_grad():
    for batch in test_loader:
        data = batch["input_ids"].to(device)
        attention_mask = batch['attention_mask'].to(device)
        label = batch["labels"].to(device)
        output = model(data, attention_mask=attention_mask, labels=label)
        predictions = torch.argmax(output.logits, dim=-1)
        test_predictions.extend(predictions.cpu().numpy())

Finally, pick an example sentence and examine the prediction. Does the prediction look correct?

Remember that 1 is the positive class, and 0 is the negative class.

In [27]:
k = 13
print(f'Text: {test_texts[k]}')
print(f'Prediction: {test_predictions[k]}')

Text: Norton 360 is the best protection you can get for your computer,will not slow down your computer and runs in the background protecting your computer and backing up all your precious information!!!!!!!
Prediction: 1


<div style="border: 4px solid coral; text-align: center; margin: auto;">
    <h3><i>Try it yourself!</i></h3>
    <br>
    <p style="text-align:center; margin:auto;"><img src="images/challenge.png" alt="Challenge" width="100" /> </p>
    <p style="margin: auto; text-align: center; margin: auto;">You trained the model for 10 epochs. Would you get better results from the validation dataset if the model trained for a longer time?</p> <br>
    <p style="margin: auto; text-align: center; margin: auto;">Note your last <code>Val_loss</code> result.</p>  <br>
    <p style="margin: auto; text-align: center; margin: auto;">Then, in the <a href="#Training-and-testing-the-model">Training and testing the model</a> section, change the <code>num_epochs</code> parameter to 20.</p> <br>
    <p style="margin: auto; text-align: center; margin: auto;">Finally, rerun the code blocks to load the pretrained model and train the model.</p> <br>
    <p style="margin: auto; text-align: center; margin: auto;">Did <code>Val_loss</code> improve?</p>
    </ol></p>
</div>

----
## Conclusion

In this lab, you learned how to import a pretrained transformer model and fine-tune it for a specific task. Although you used a lighter version of the BERT model, these types of models tend to use large amounts of compute power. For that reason, you worked with only the first 2,000 data points of the dataset. To see more general results, you would need to spend more time training with the whole dataset.

---
## Next lab
In the next lab, you will learn how to read images and plot them as you start to learn about computer vision (CV) problems.